In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-2: Preferences, reward models and RLHF

## Where this session fits

> **The limitation we start from: for open-ended tasks no program can check a completion, and people find it hard to give a score.**

- **The question:** how does a reward come from human comparisons, and how is it used?
- **Position: session 2 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-1 mapped post-training and named the three sources of a reward
    - C1 gave a one-slide gloss on reward models; this session is the full story
    - the next session, RLLM-3, replaces the human labeler by a model
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. say what preference data is and why comparisons are often easier than scores
2. state the Bradley-Terry model, and compute a probability and a loss from two scores
3. say how a reward model is built and trained
4. read the three stages of RLHF, and say where the KL reward and the reference model enter
5. explain reward over-optimization, and what limits it


# Preferences versus rewards

**Why this section?** The reward of Parts A and B was a number from the environment. Some problems offer only a ranking.

> **A reward is a number whose size means something. A preference is a comparison: which of two is better.**

- a **reward** is a number: any two outputs can be compared, and the gap means something
- a **preference** says "A is better than B", usually for only some pairs, with no magnitude
- style is a preference problem
    - prompt: "How do I change a tire?"
    - reply A: an accurate step-by-step answer
    - reply B: a correct but terse answer without jack placement or tightening order
    - both are acceptable, but most people prefer A


## Preference data

**Why this section?** The unit of data for everything that follows.

> **A preference example is a triple: a prompt, a preferred completion, and a non-preferred completion.**

- a **preference example** is $(\x, \y^+, \y^-)$
    - the prompt $\x$
    - the preferred completion $\y^+$
    - the non-preferred completion $\y^-$
- both completions may be *acceptable*: the data records an ordering, not correctness
- why comparisons can be easier to collect than scores
    - raters disagree about the scale of a score (a 7 or an 8?)
    - they agree more often about which of two is better
    - people disagree on a share of pairs, so the labels are noisy
- the price: a comparison carries less information than a score, and raters can still disagree
- the data can be used in two ways
    - imitate the preferred completion (SFT)
    - or **learn a reward from the contrast**, and optimize it with RL (this session)


## From comparisons to a reward: the Bradley-Terry model

**Why this section?** How a pile of comparisons becomes a reward model.

> **Bradley-Terry: the probability that $\y^+$ is preferred is the sigmoid of the difference of two scores. Fit the scores by maximum likelihood.**

- the **reward model** $\rew_\psi(\x, \y)$ is a network with parameters $\psi$
    - it gives one score to a whole completion, not a per-step reward: **the score** is its output, read at the last token
    - $\psi$, not $\phi$: B3 used $\phi$ for the critic (papers often write $\phi$ for both)
- the **Bradley-Terry model**
    - $P(\y^+ \succ \y^-) = \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
    - $\sigma(z) = 1/(1 + e^{-z})$ is the sigmoid, and $\succ$ reads "is preferred to"
- the loss, averaged over the data: $-\log \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
    - it is a logistic-regression loss on the score difference
- **the classifier view:** a reward model is a classifier for "which of the two completions is preferred?"
    - the input is the pair $(\y^+, \y^-)$ for a prompt $\x$, and the label is which one won
    - the score difference is the logit, and the sigmoid turns it into a probability
    - the loss is the usual binary cross-entropy, so fitting $\psi$ is ordinary classification
    - the network scores each completion alone; only the training compares two

**Bradley-Terry by numbers**

- the model scores $\y^+$ at $2.0$ and $\y^-$ at $0.5$
    - the difference is $1.5$, so $P(\y^+ \succ \y^-) = \sigma(1.5) \approx 0.82$
    - the loss is $-\log 0.82 \approx 0.20$
- the reverse order, a difference of $-1.5$: $\sigma(-1.5) \approx 0.18$ and the loss is about $1.70$

<img src="images/rllm2_bradley_terry.png" width=75%>

- only **differences** matter: adding a constant to every score of a prompt changes nothing
    - so comparisons need no common scale, and the reward has no absolute zero
    - across prompts the offset is arbitrary, which matters when scores meet a KL term
- the loss keeps falling as the gap widens, so a reward model can overfit its noisy labels
    - so reward models are trained briefly or regularized


## Building a reward model

**Why this section?** The reward model is itself a trained network, with choices that matter.

> **A reward model is usually a language model with a scalar output, trained briefly on comparisons.**

- **architecture:** a language model (often the SFT model) with a scalar head in place of the next-token output
    - it reads the prompt and the completion, and outputs one number
- **data:** several completions per prompt, ranked by raters
    - a ranking of $K$ completions gives $K(K-1)/2$ pairs
- **training:** the Bradley-Terry loss on those pairs
    - few passes over the data: it overfits quickly
- **what it does not do:** it does not know the truth, only what raters tended to prefer
- known weaknesses
    - it can prefer long answers, or a confident tone
    - it is only reliable near the completions it was trained on


# RLHF: the three stages

**Why this section?** The route from comparisons back to the Part B machinery.

> **RLHF has three stages: supervised fine-tuning, a reward model, and RL against that reward model with a KL term to the SFT model.**

<img src="images/rllm2_rlhf.png" width=95%>

- **RLHF** is reinforcement learning from human feedback
- stage 3 needs only prompts: no labeled completions

**Stage 3 is the loop of C1 and C2**

- sample completions from the current policy, score each with $\rew_\psi$
- credit
    - PPO: a critic and GAE (B3), with the per-token KL reward of C1
    - GRPO: advantages from a group (C2), with the KL as a loss term
- update with the clipped step (B4)
- nothing new in the optimizer: the reward model is one more source of reward
- the cost: models in memory
    - the policy, the reference and the reward model
    - plus the old policy and a critic if PPO is used


## Reward over-optimization

**Why this section?** The reward model is a proxy. Pushing hard on it can make the real quality worse.

> **A reward model is trusted only near its training data. Optimized hard, the policy finds completions it over-rates, and real quality falls.**

<img src="images/rllm2_overoptimization.png" width=80%>

- (the curves are invented, to show the shape)
- the score the policy is trained on keeps rising
- the quality people would give peaks, then falls

**What limits over-optimization**

    - the **KL term** to the reference model keeps the policy near it, and so near where the reward model was trained
    - training for fewer steps, or with a smaller learning rate
    - a larger or better reward model, or several of them
- a check: sample completions late in training and read them


## An example preference dataset

**Why this section?** A concrete case, so the triple is not abstract.

> **UltraFeedback (binarized) is a dataset of (prompt, chosen, rejected) triples, labeled by AI feedback rather than human feedback.**

- [UltraFeedback (binarized)](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized): columns *prompt*, *chosen*, *rejected*
- how it was built
    - prompts gathered from many sources
    - completions to each prompt from several different language models
    - a strong language model (GPT-4) acts as the annotator
        - it critiques each completion on instruction following, truthfulness, honesty and helpfulness
        - its scores rank the completions
- the **binarized** version keeps one pair per prompt
    - the best-scored completion is *chosen*
    - one of the others, at random, is *rejected*
- it is **AI feedback**: the next session, RLLM-3, studies that choice
- the completions come from other models, so the pairs are off-policy for the model being trained


# What we established, and what it sets up

- **Established**
    - preference data records comparisons $(\x, \y^+, \y^-)$, often easier to collect than scores
    - Bradley-Terry: the probability of a preference is the sigmoid of a score difference
    - a reward model is a language model with a scalar head, trained briefly on comparisons
    - RLHF: SFT, a reward model, then RL with the KL term to the reference
    - a reward model is a proxy: over-optimized, it misleads the policy
- **Sets up**
    - **the limitation:** human labels are costly and slow
    - next session, RLLM-3: AI feedback (RLAIF) and Constitutional AI
    - RLLM-10 treats DPO, which uses the same data without a reward model
